In [13]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict,Annotated
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.messages import BaseMessage,HumanMessage
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langgraph.graph.message import add_messages
from dotenv import load_dotenv
from langchain_community.vectorstores import FAISS
from langchain_core.tools import tool
from langgraph.prebuilt import ToolNode,tools_condition

In [8]:
load_dotenv()
llm=ChatGroq(model='llama-3.1-8b-instant')

In [14]:
loader=PyPDFLoader(r"C:\Users\KhushiAgarwal\OneDrive - Aditi Consulting\Downloads\intro-to-ml.pdf")
docs=loader.load()
len(docs)

392

In [17]:
splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
chunks=splitter.split_documents(docs)
len(chunks)

973

In [18]:
embeddings=HuggingFaceEmbeddings(model='all-MiniLM-L6-v2')

c:\Users\KhushiAgarwal\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\KhushiAgarwal\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 605

In [19]:
vector_store=FAISS.from_documents(chunks,embeddings)

In [20]:
retriever=vector_store.as_retriever(search_type='similarity',search_kwargs={'k':4})

In [21]:
@tool
def rag_tool(query):
    '''
    Retrieve relevant information from the pdf document.
    Use this tool when the user asks factual/conceptual questions
    that might be answered from the stored documents.
    '''
    result=retriever.invoke(query)
    context=[doc.page_content for doc in result]
    metadata=[doc.metadata for doc in result]

    return {
        'query':query,
        'context':context,
        'metadata':metadata
    }

In [22]:
retriever.invoke('what is decision tree')

[Document(id='6108cd88-b609-4626-a1ed-ab931f229f41', metadata={'producer': '3-Heights(TM) PDF Optimization Shell 5.9.1.5 (http://www.pdf-tools.com)', 'creator': 'AH CSS Formatter V6.2 MR4 for Linux64 : 6.2.6.18551 (2014/09/24 15:00JST)', 'creationdate': '2016-09-21T13:04:39+00:00', 'author': 'Andreas C. Müller and Sarah Guido', 'title': 'Introduction to Machine Learning with Python', 'trapped': '/False', 'moddate': '2020-08-19T07:09:16+02:00', 'source': 'C:\\Users\\KhushiAgarwal\\OneDrive - Aditi Consulting\\Downloads\\intro-to-ml.pdf', 'total_pages': 392, 'page': 87, 'page_label': '74'}, page_content='Figure 2-26. Decision boundary of tree with depth 9 (left) and part of the corresponding\ntree (right); the full tree is quite large and hard to visualize\nA prediction on a new data point is made by checking which region of the partition\nof the feature space the point lies in, and then predicting the majority target (or the\nsingle target in the case of pure leaves) in that region. Th

In [24]:
tools=[rag_tool]
llm_with_tools=llm.bind_tools(tools)

In [26]:
class ChatState(TypedDict):
    messages:Annotated[list[BaseMessage],add_messages]

In [27]:
def chat_node(state:ChatState):
    messages=state['messages']
    response=llm_with_tools.invoke(messages)
    return {'messages':[response]}

In [28]:
tool_node=ToolNode(tools)

In [31]:
graph=StateGraph(ChatState)
graph.add_node('chat_node',chat_node)
graph.add_node('tools',tool_node)
graph.add_edge(START,'chat_node')
graph.add_conditional_edges('chat_node',tools_condition)
graph.add_edge('tools','chat_node')
chatbot=graph.compile()


In [34]:
result=chatbot.invoke(
    {
        'messages':[
            HumanMessage(
                content=(
                    'using the pdf notes,explain how to split a node in a decision tree'
                )
            )
        ]
    }
)

In [ ]:
print(result['messages'][-1].content)

To split a node in a decision tree, the algorithm searches over all possible tests and finds the one that is most informative about the target variable. This is done by testing whether a particular feature is less than or equal to a certain value, and then splitting the data into two regions based on this test. The region that is most informative about the target variable is then further split, and this process is repeated until each region in the partition contains data points that all share the same target value.


: 